# Musicm8 v8 — score-controlled vocals

Lead vocals now use **SoulX-Singer** with explicit words/phonemes, note pitches and word durations. ACE-Step is no longer the main lead singer. Paste your own lyrics into `CUSTOM_LYRICS`. Optionally set `VOICE_REFERENCE` to a clean 5–30 second voice/singing recording that you own or have permission to use; SoulX-SVC will convert the score-correct guide into that timbre. Leave it blank for the default singer.

In [ ]:
# ============================================================
# MUSICM8 V8 — ONE CLICK COMPLETE SONG
# ============================================================
import os, sys, json, shutil, subprocess, secrets
from pathlib import Path

IDEA = "dark UK garage song about knowing a relationship is over but not being able to leave, emotional chords, deep moving bass"
BARS = 32
FIXED_SEED = None
SEED = int(FIXED_SEED) if FIXED_SEED is not None else secrets.randbelow(2_000_000_000)
print(f"🎲 MUSICM8 SONG SEED: {SEED}")

# Paste your exact words here. Leave blank for AI lyrics.
CUSTOM_LYRICS = r"""
""".strip()

# Optional voice clone/timbre reference. Use only a voice you own or have permission to use.
# Best input: 5-30 seconds of clean, mostly dry singing. Leave "" for default SoulX singer.
VOICE_REFERENCE = ""
# Example: VOICE_REFERENCE = "/content/drive/MyDrive/Musicm8/voice_reference.wav"

VOCALS = True
VOCAL_STEPS = 24
MATCH_ITERS = 48
MATCH_SECONDS = 3.0
FORCE_SOUND_MATCH = False
AI_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Musicm8")
WORK = ROOT / "work"
AUDIO = ROOT / "audio"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
WORK.mkdir(parents=True, exist_ok=True); AUDIO.mkdir(parents=True, exist_ok=True)

if (REPO / ".git").exists():
    subprocess.run(["git","-C",str(REPO),"fetch","--depth","1","origin","main"], check=True)
    subprocess.run(["git","-C",str(REPO),"reset","--hard","origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git","clone","--depth","1",REPO_URL,str(REPO)], check=True)

os.chdir(REPO)
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements-ai.txt"], check=True)
subprocess.run(["apt-get","update","-qq"], check=True)
subprocess.run(["apt-get","install","-y","-qq","espeak-ng","ffmpeg"], check=False)

import torch
if not torch.cuda.is_available(): raise RuntimeError("No GPU connected. Runtime → Change runtime type → GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

lyrics_input = WORK / "user_lyrics_input.txt"
if CUSTOM_LYRICS.strip():
    lyrics_input.write_text(CUSTOM_LYRICS.strip()+"\n", encoding="utf-8")
    print("✍️ USER LYRICS MODE")
else:
    if lyrics_input.exists(): lyrics_input.unlink()
    print("✍️ AI LYRICS MODE")

cmd=[sys.executable,"-u","ai_producer_workflow.py","--root",str(ROOT),"--repo",str(REPO),"--idea",IDEA,"--bars",str(BARS),"--seed",str(SEED),"--ai-model",AI_MODEL,"--match-iters",str(MATCH_ITERS),"--match-seconds",str(MATCH_SECONDS),"--vocal-steps",str(VOCAL_STEPS)]
if CUSTOM_LYRICS.strip(): cmd += ["--lyrics-file",str(lyrics_input)]
if VOICE_REFERENCE.strip(): cmd += ["--voice-reference",VOICE_REFERENCE.strip()]
if not VOCALS: cmd.append("--no-vocals")
if FORCE_SOUND_MATCH: cmd.append("--force-sound-match")
subprocess.run(cmd, check=True)

PROJECT=WORK/"ai_projects/latest"
(PROJECT/"song_seed.txt").write_text(str(SEED),encoding="utf-8")

print("\n================ 📝 LYRICS USED ================\n")
lf=PROJECT/"lyrics.txt"
print(lf.read_text(encoding="utf-8") if lf.exists() else "No lyrics file")
print("=================================================\n")

for report_name in ("vocal_quality.json","vocal_word_quality.json"):
    rp=PROJECT/"vocals"/report_name
    if rp.exists():
        data=json.loads(rp.read_text())
        print("✅",report_name, json.dumps(data,indent=2,ensure_ascii=False))

from IPython.display import Audio, display
for label,p in [
    ("INSTRUMENTAL",PROJECT/"master_instrumental.wav"),
    ("SOULX SCORE GUIDE",PROJECT/"vocals/soulx_score_guide.wav"),
    ("VOICE-CLONED VOCAL",PROJECT/"vocals/soulx_voice_cloned.wav"),
    ("QA-PASSED LEAD",PROJECT/"vocals/neural_lead_synced.wav"),
    ("VOCAL MIX",PROJECT/"vocals/vocal_mix.wav"),
    ("FINAL SONG",PROJECT/"master.wav")]:
    if p.exists():
        print("\n🎵",label)
        display(Audio(str(p)))
print(f"\n🌱 Seed: {SEED}")


## 🎤 Vocal-only retry

Use this when the instrumental already exists. It rebuilds only the **word-by-word vocal melody**, generates SoulX singing, checks pitch and lyric intelligibility, and remixes the vocal. It does not regenerate the music.

In [ ]:
# MUSICM8 V8 — VOCAL ONLY
import os, sys, json, subprocess
from pathlib import Path
from IPython.display import Audio, display
ROOT=Path('/content/drive/MyDrive/Musicm8'); REPO=Path('/content/Musicm8'); PROJECT=ROOT/'work/ai_projects/latest'
VOICE_REFERENCE = ""  # optional authorized clean voice/singing reference
VOCAL_STEPS=24
seed_file=PROJECT/'song_seed.txt'; SEED=int(seed_file.read_text().strip()) if seed_file.exists() else 42
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin','main'],check=True)
subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
os.chdir(REPO)
cmd=[sys.executable,'-u','retry_vocals.py','--root',str(ROOT),'--repo',str(REPO),'--steps',str(VOCAL_STEPS),'--seed',str(SEED)]
if VOICE_REFERENCE.strip(): cmd += ['--voice-reference',VOICE_REFERENCE.strip()]
subprocess.run(cmd,check=True)
print('\n================ 📝 LYRICS USED ================\n'); print((PROJECT/'lyrics.txt').read_text()); print('=================================================\n')
for rp in [PROJECT/'vocals/vocal_quality.json',PROJECT/'vocals/vocal_word_quality.json']:
    if rp.exists(): print(rp.name, json.dumps(json.loads(rp.read_text()),indent=2,ensure_ascii=False))
for label,p in [('SOULX GUIDE',PROJECT/'vocals/soulx_score_guide.wav'),('VOICE CLONE',PROJECT/'vocals/soulx_voice_cloned.wav'),('QA-PASSED LEAD',PROJECT/'vocals/neural_lead_synced.wav'),('FINAL SONG',PROJECT/'master.wav')]:
    if p.exists(): print('\n🎵',label); display(Audio(str(p)))


## Diagnostics
If SoulX fails, this prints the actual backend tail and status.

In [ ]:
from pathlib import Path
project=Path('/content/drive/MyDrive/Musicm8/work/ai_projects/latest')
status=project/'vocals/vocal_status.json'; log=project/'vocals/vocal_backend.log'
print(status.read_text() if status.exists() else 'No vocal_status.json')
if log.exists(): print('\n--- SOULX BACKEND LOG TAIL ---\n'+'\n'.join(log.read_text(errors='ignore').splitlines()[-160:]))
